## Cars' Price Predictor using **XGBoosting**

### Libs importing

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
import pickle
import os

In [3]:
df = pd.read_csv('../Daten/autoscout24-germany-dataset.csv')
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (46405, 9)


### Clean the Data

In [3]:
df_clean = df.dropna()
df_clean = df_clean[
    (df_clean['price'] > 500) & (df_clean['price'] < 100000) &
    (df_clean['mileage'] >= 0) & (df_clean['mileage'] < 500000) &
    (df_clean['hp'] > 0) & (df_clean['hp'] < 1000) &
    (df_clean['year'] >= 1990) & (df_clean['year'] <= 2023)
]

# Remove price outliers
Q1, Q3 = df_clean['price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df_clean = df_clean[(df_clean['price'] >= Q1 - 1.5*IQR) & (df_clean['price'] <= Q3 + 1.5*IQR)]
print(f"Cleaned dataset shape: {df_clean.shape}")

Cleaned dataset shape: (42530, 9)


### Feature Engineering (add a new feature)

In [4]:
df_features = df_clean.copy()

current_year = 2023
df_features['age'] = current_year - df_features['year']
df_features['mileage_per_year'] = df_features['mileage'] / (df_features['age'] + 1)
df_features['hp_per_year'] = df_features['hp'] / (df_features['age'] + 1)
df_features['log_mileage'] = np.log1p(df_features['mileage'])

df_features['age_category'] = pd.cut(df_features['age'], 
                                   bins=[0, 3, 7, 12, float('inf')],
                                   labels=['New', 'Recent', 'Mature', 'Old'])
df_features['mileage_category'] = pd.cut(df_features['mileage'], 
                                       bins=[0, 50000, 100000, 150000, 200000, float('inf')],
                                       labels=['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh'])

### Encode categorical variables and split them into test and train data

In [5]:
label_encoders = {}
categorical_cols = ['make', 'model', 'fuel', 'gear', 'offerType', 'age_category', 'mileage_category']

df_model = df_features.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# Select features
feature_columns = [
    'mileage', 'hp', 'year', 'age', 'mileage_per_year', 'hp_per_year', 'log_mileage',
    'make_encoded', 'model_encoded', 'fuel_encoded', 'gear_encoded', 'offerType_encoded',
    'age_category_encoded', 'mileage_category_encoded'
]

X = df_model[feature_columns]
y = df_model['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=37)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

Training set: (34024, 14), Test set: (8506, 14)


### Train the first version of the Model

In [6]:
erst_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=37)
erst_model.fit(X_train, y_train)
y_pred_first = erst_model.predict(X_test)

erst_r2 = r2_score(y_test, y_pred_first)
erst_rmse = np.sqrt(mean_squared_error(y_test, y_pred_first))
print(f"First Model - R²: {erst_r2:.4f}, RMSE: {erst_rmse:.0f}")

First Model - R²: 0.9460, RMSE: 1784


### Function to Optimize The Model  with Optuna (Bayesian Optimization)

In [7]:
def optimizer(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [200, 300, 400]),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2),
        'random_state': 37,
        'n_jobs': -1
    }

    model = xgb.XGBRegressor(**params)
    score = -cross_val_score(model, X_train, y_train, cv=3, scoring='neg_mean_squared_error', n_jobs=-1).mean()
    return score

### Get best parameters and re-train the model (with best options)

In [16]:
study = optuna.create_study(direction='minimize')
study.optimize(optimizer, n_trials=70)

# Get best parameters
best_params = study.best_params
best_value = study.best_value

print(f"Best parameters: {best_params}")
print(f"Best RMSE: {np.sqrt(best_value):.2f}")

best_model = xgb.XGBRegressor(**best_params, random_state=37, n_jobs=-1)
best_model.fit(X_train, y_train)

[I 2025-06-11 02:36:14,660] A new study created in memory with name: no-name-7b13db22-fe1a-4e72-ba85-e82c5ee431c3
[I 2025-06-11 02:36:17,607] Trial 0 finished with value: 3380968.25 and parameters: {'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.1307273539522444, 'subsample': 0.7749113147608142, 'colsample_bytree': 0.8065822343698243, 'reg_alpha': 0.7652144680744758, 'reg_lambda': 1.8965898343874659}. Best is trial 0 with value: 3380968.25.
[I 2025-06-11 02:36:18,467] Trial 1 finished with value: 3426527.75 and parameters: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.07208858452532951, 'subsample': 0.9748842942766517, 'colsample_bytree': 0.8043449760877964, 'reg_alpha': 0.9879849982272514, 'reg_lambda': 0.6916893823205509}. Best is trial 0 with value: 3380968.25.
[I 2025-06-11 02:36:19,338] Trial 2 finished with value: 3403536.25 and parameters: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.07306993863030255, 'subsample': 0.8590434911552081, 'colsample_b

Best parameters: {'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.09897321130220113, 'subsample': 0.8346102414877049, 'colsample_bytree': 0.6462250170525763, 'reg_alpha': 0.49638580723677267, 'reg_lambda': 1.6362975243563844}
Best RMSE: 1774.30


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6462250170525763
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
y_pred_optimized = best_model.predict(X_test)

optimized_r2 = r2_score(y_test, y_pred_optimized)
optimized_rmse = np.sqrt(mean_squared_error(y_test, y_pred_optimized))
optimized_mae = mean_absolute_error(y_test, y_pred_optimized)
mape = np.mean(np.abs((y_test - y_pred_optimized) / y_test)) * 100

print(f"\nFinal Model Performance:")
print(f"R² Score: {optimized_r2:.4f}")
print(f"RMSE: {optimized_rmse:.0f} €")
print(f"MAE: {optimized_mae:.0f} €")
print(f"MAPE: {mape:.1f}%")

print(f"\nImprovement: R² +{optimized_r2 - erst_r2:.4f}, RMSE -{erst_rmse - optimized_rmse:.0f}")



Final Model Performance:
R² Score: 0.9530
RMSE: 1654 €
MAE: 1177 €
MAPE: 10.3%

Improvement: R² +0.0070, RMSE -130


### Saving the model and the encoders (Cause it was better than any available Online) 😁

In [15]:
os.makedirs('../models/xgb', exist_ok=True)

# Save model
with open('../models/xgb/xgb_car_price_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save label encoders
with open('../models/xgb/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Save feature columns
with open('../models/xgb/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("Model and encoders saved successfully!")

Model and encoders saved successfully!
